# Initialization

## Import Libraries

In [1]:
import numpy as np
import pandas as pd
from numba import njit, prange
import math

import matplotlib.pyplot as plt
import os
from aeon.utils.numba.general import z_normalise_series_2d
from aeon.distances import get_distance_function
import time

In [2]:
# %load_ext autoreload
# %autoreload 2
from ksfdtw.distance_measures import (
    # euclidean_distance as ksfdtw_euclidean_distance,
    # dtw as ksfdtw_dtw,
    usdtw_prime as usdtw_prime,
    psdtw_prime_vanilla as psdtw_prime_vanilla,
    psdtw_prime_parallel as psdtw_prime_parallel,
    psdtw_prime_parallel_bsf as psdtw_prime_parallel_bsf,
    psdtw_prime_parallel_bsf_lb as psdtw_prime_parallel_bsf_lb,
    psdtw_prime_parallel_bsf_lb2 as psdtw_prime_parallel_bsf_lb2,
    psdtw_prime_parallel_bsf_lb3 as psdtw_prime_parallel_bsf_lb3,
    cut_based_distance as cut_based_distance,
)
from ksfdtw.utils import precision_at_k, nearest_neighbor_search
# from aeon.distances import (
#     euclidean_distance as aeon_euclidean_distance,
#     squared_distance as aeon_squared_distance,
#     dtw_distance as aeon_dtw_distance,
# )

## Import Dataset

In [3]:
# The ten datasets used in the experiments:
# SonyAIBORobotSurface1
# ECG200
# MedicalImages
# CBF
# SwedishLeaf
# Plane
# PowerCons
# GunPoint
# Adiac
# Epilepsy

dataset_name = "GunPoint"
# Target set
P = 3
l = 1.50
data = np.load(
    f"../data_processed/{dataset_name}_P{P}_uniform.npz",
    allow_pickle=True,
)
# Use Train or Test set or Both?
trans_uniform_concatenated = data["X_train_trans_uniform_concatenated"]
# trans_uniform_concatenated = data["X_test_trans_uniform_concatenated"]
# trans_uniform_concatenated = np.concatenate((data["X_train_trans_uniform_concatenated"], data["X_test_trans_uniform_concatenated"]), axis=0)

# Query set
# P = 3
data = np.load(
    f"../data_processed/{dataset_name}_P{P}_l{l:.2f}_random.npz",
    allow_pickle=True,
)
# Use Train or Test set or Both?
trans_random_concatenated = data["X_train_trans_random_concatenated"]
# trans_random_concatenated = data["X_test_trans_random_concatenated"]
# trans_random_concatenated = np.concatenate((data["X_train_trans_random_concatenated"], data["X_test_trans_random_concatenated"]), axis=0)


## Z-normalise the transformed series

In [4]:
trans_uniform_concatenated = z_normalise_series_2d(trans_uniform_concatenated)
trans_random_concatenated = z_normalise_series_2d(trans_random_concatenated)

## Plot graph

In [5]:
instance_idx = 0

In [6]:
# plt.plot(trans_uniform_concatenated[instance_idx, :-1], color="red")
# plt.plot(trans_random_concatenated[instance_idx, :-1], color="blue", linestyle="--")
# plt.axis("off")
# plt.tight_layout()
# # plt.savefig("../figures/concatenation_uniform_vs_random.pdf", format="pdf", bbox_inches="tight")
# plt.show()
# print("Length of the instance: ", len(trans_uniform_concatenated[instance_idx]) - 1)
# print("Number of instances in the query set (target set): ", len(trans_uniform_concatenated))

In [7]:
# Create indices (0 to N-1)
indices = np.arange(len(trans_uniform_concatenated)).reshape(-1, 1)

# Append indices
trans_uniform_concatenated = np.hstack((trans_uniform_concatenated, indices))
trans_random_concatenated = np.hstack((trans_random_concatenated, indices))


In [8]:
# Assign query and target sets
# Query set
query_set = trans_random_concatenated

# Target set
target_set = trans_uniform_concatenated
if len(query_set) != len(target_set):
    raise ValueError("query_set and target_set have different sizes!")

# Searching with distance measures provided in aeon

## Precision@k

Compute $P@k$ for querying $Q \in$ `query_set` using `method_name` on `target_set`

In [9]:
# # https://www.aeon-toolkit.org/en/latest/api_reference/distances.html
# # https://www.aeon-toolkit.org/en/stable/api_reference/auto_generated/aeon.distances.get_distance_function.html
# dist_funcs = {
#     "squared": get_distance_function("squared"),  # ED
#     "dtw": lambda Q, C: get_distance_function("dtw")(Q, C, window=0.1),
#     "adtw": lambda Q, C: get_distance_function("adtw")(Q, C, window=0.1),
#     "ddtw": lambda Q, C: get_distance_function("ddtw")(Q, C, window=0.1),
#     # "erp": lambda Q, C: get_distance_function("erp")(Q, C, window=0.1),
#     # "edr": lambda Q, C: get_distance_function("edr")(Q, C, window=0.1),
#     # "lcss": lambda Q, C: get_distance_function("lcss")(Q, C, window=0.1),
#     # "manhattan": get_distance_function("manhattan"),
#     # "minkowski": get_distance_function("minkowski"),
#     # "msm": lambda Q, C: get_distance_function("msm")(Q, C, window=0.1),
#     # "sbd": get_distance_function("sbd"),
#     "shape_dtw": lambda Q, C: get_distance_function("shape_dtw")(Q, C, window=0.1),
#     # "twe": lambda Q, C: get_distance_function("twe")(Q, C, window=0.1),
#     "wddtw": lambda Q, C: get_distance_function("wddtw")(Q, C, window=0.1),
#     "wdtw": lambda Q, C: get_distance_function("wdtw")(Q, C, window=0.1),
# }

In [10]:
# for dist_name, dist_func in dist_funcs.items():
#     precision_at_1, precision_at_3, precision_at_5, precision_at_7 = 0, 0, 0, 0
#     for i in range(0, len(query_set)):

#         query_feat = query_set[i][:-1] # The actual time series data
#         query_idx = query_set[i][-1] # Index
        
#         distances = np.array([dist_func(query_feat, x[:-1]) for x in target_set])

#         true_match_pos = np.where(target_set[:, -1] == query_idx)[0][0]
#         # print(f"Query index: {query_idx}, True match position in target set: {true_match_pos}")

#         precision_at_1 += precision_at_k(distances, true_match_pos, 1)
#         precision_at_3 += precision_at_k(distances, true_match_pos, 3)
#         precision_at_5 += precision_at_k(distances, true_match_pos, 5)
#         precision_at_7 += precision_at_k(distances, true_match_pos, 7)
#     print(
#         # f"{precision_at_1 / len(query_set):.2f}",
#         f"{precision_at_1 / len(query_set):.2f} & {precision_at_3 / len(query_set):.2f}",
#         end=" & ",
#     )

# Searching with PSED, PSDTW

In [11]:
# # psdtw_prime_vanilla, psdtw_prime_parallel, psdtw_prime_parallel_bsf, psdtw_prime_parallel_bsf_lb, psdtw_prime_parallel_bsf_lb2
# dist_method = 0  # 0 for ED, 1 for DTW, 15 for DTW (self-defined)
# function_used = psdtw_prime_vanilla

# dist_func_pp = lambda Q, C: function_used(
#         Q, C, l=l, r=0.0, P=P, dist_method= dist_method
#     )
# dist_func_p = lambda Q, C: dist_func_pp(Q, C)

In [12]:
# def dist_func(Q, C):
#     dist, _, _ = dist_func_p(Q, C)
#     return dist

In [13]:
# # Warmup for numba
# dist_func(
#     trans_uniform_concatenated[instance_idx][:-1],
#     trans_random_concatenated[instance_idx][:-1],
# )
# start = time.time()
# dist_func(
#     trans_uniform_concatenated[instance_idx][:-1],
#     trans_random_concatenated[instance_idx][:-1],
# )
# end = time.time()
# elapsed_time = end - start
# # print("Elapsed time for a single distance computation: " + str(elapsed_time))

## Precision@k

In [14]:
# all_distances = []
# all_count_dist_calls = []
# all_cuts = []
# start = time.time()
# precision_at_1, precision_at_3, precision_at_5, precision_at_7 = 0, 0, 0, 0
# for i in range(0, len(query_set)):
#     results = [dist_func_p(query_set[i][:-1], x[:-1]) for x in target_set] # Strip index
#     dist_arr, count_dist_calls_arr, cuts_arr = zip(*results)
#     distances = np.array(dist_arr)

#     # store per-iteration results
#     all_distances.append(distances)
#     all_count_dist_calls.append(count_dist_calls_arr)
#     all_cuts.append(cuts_arr)

#     # Find true match position
#     query_idx = query_set[i][-1]
#     target_idxs = target_set[:, -1]
#     true_match_pos = np.where(target_idxs == query_idx)[0][0]

#     precision_at_1 += precision_at_k(distances, true_match_pos, 1)
#     precision_at_3 += precision_at_k(distances, true_match_pos, 3)
#     # precision_at_5 += precision_at_k(distances, true_match_pos, 5)
#     # precision_at_7 += precision_at_k(distances, true_match_pos, 7)
# print(
#     f"{precision_at_1 / len(query_set):.2f} & {precision_at_3 / len(query_set):.2f}",
#     end=" & ",
# )
# end = time.time()
# elapsed_time = end - start
# print()
# print("Elapsed time: " + str(elapsed_time))
# print("Average Elapsed time: " + str(elapsed_time / len(query_set)))

# total_count_dist_calls = 0
# for r in all_count_dist_calls:
#     total_count_dist_calls += np.sum(r)
# print("Total distance measure calls: " + str(total_count_dist_calls))
# total_count_dist_calls_original = total_count_dist_calls

In [15]:
# os.makedirs("../outputs", exist_ok=True)
# np.savez(
#     f"../outputs/{dataset_name}_P{P}_l{l:.2f}_dist_method{dist_method}_{function_used.__name__}.npz",
#     all_distances=np.array(all_distances, dtype=object),
#     all_count_dist_calls=np.array(all_count_dist_calls, dtype=object),
#     all_cuts=np.array(all_cuts, dtype=object),
#     precision_at_1=precision_at_1 / len(query_set),
#     precision_at_3=precision_at_3 / len(query_set),
#     precision_at_5=precision_at_5 / len(query_set),
#     precision_at_7=precision_at_7 / len(query_set),
#     elapsed_time=elapsed_time,
# )

# Searching with Cut-based distance

## Import computed cuts

In [16]:
# dist_method = 0  # 0 for ED, 1 for DTW, 15 for DTW (self-defined)
# # function_used: psdtw_prime_vanilla, psdtw_prime_parallel
# function_used = psdtw_prime_vanilla

In [17]:
# data = np.load(
#     f"../outputs/{dataset_name}_P{P}_l{l:.2f}_dist_method{dist_method}_{function_used.__name__}.npz",
#     allow_pickle=True,
# )

# all_distances = np.ascontiguousarray(data["all_distances"], dtype=np.float64)
# all_count_dist_calls = np.ascontiguousarray(
#     data["all_count_dist_calls"], dtype=np.float64
# )
# all_cuts = np.ascontiguousarray(data["all_cuts"], dtype=np.float64)

In [18]:
# print("Imported results from file:")
# # Results of PSD
# print(
#     f"{data["precision_at_1"]:.2f}",
#     f"{data["precision_at_3"]:.2f}",
#     end=" & ",
#     # f"{data["precision_at_5"]:.2f}",
#     # f"{data["precision_at_7"]:.2f}",
# )
# # print(
# #     f"{precision_at_1 / len(query_set):.2f} & {precision_at_3 / len(query_set):.2f}",
# #     end=" & ",
# # )
# print()
# print("Elapsed time:", data["elapsed_time"])

# total_count_dist_calls = 0
# for r in all_count_dist_calls:
#     total_count_dist_calls += np.sum(r)
# print("Total distance measure calls: " + str(total_count_dist_calls))
# original_elapsed_time = data["elapsed_time"]
# original_total_count_dist_calls = total_count_dist_calls

## Compute Cut-based distances

In [19]:
# # 0: aeon_squared_distance, 1: aeon_dtw_distance, 2: aeon_adtw_distance, 3: aeon_ddtw_distance, 4: aeon_erp_distance, 5: aeon_edr_distance
# # 6: aeon_lcss_distance, 7: aeon_manhattan_distance, 8: aeon_minkowski_distance, 9: aeon_msm_distance, 10: aeon_sbd_distance
# # 11: aeon_shape_dtw_distance, 12: aeon_twe_distance, 13: aeon_wddtw_distance, 14: aeon_wdtw_distance
# # for i in range(0, 15):
# for i in [1, 2, 3, 11, 13, 14]:
#     # print("dist_method: " + str(i))
#     dist_method = i
#     precision_at_1, precision_at_3, precision_at_5, precision_at_7 = 0, 0, 0, 0
#     for i in range(0, len(query_set)):
#         query_vec = query_set[i][:-1]
#         query_idx = query_set[i][-1]
#         target_idxs = target_set[:, -1]
#         true_match_pos = np.where(target_idxs == query_idx)[0][0]

#         distances = np.array(
#             [
#                 cut_based_distance(
#                     query_vec,
#                     target_set[j][:-1],
#                     0.1,
#                     l,
#                     P,
#                     dist_method=dist_method,
#                     cuts=all_cuts[i][j],
#                 )
#                 for j in range(0, len(target_set))
#             ]
#         )
#         precision_at_1 += precision_at_k(distances, true_match_pos, 1)
#         precision_at_3 += precision_at_k(distances, true_match_pos, 3)
#     print(
#         f"{precision_at_1 / len(query_set):.2f} & {precision_at_3 / len(query_set):.2f}",
#         end=" & ",
#     )

# Nearest neighbor search with bsf

In [20]:
# Shuffle Transformed Uniform (Target Set)
np.random.seed(42)
np.random.shuffle(trans_uniform_concatenated)

# Shuffle Transformed Random (Query Set)
np.random.seed(43)
np.random.shuffle(trans_random_concatenated)

print("Shuffled.")
print("New shape:", trans_uniform_concatenated.shape)

# Assign query and target sets
# Query set
query_set = trans_random_concatenated

# Target set
target_set = trans_uniform_concatenated
if len(query_set) != len(target_set):
    raise ValueError("query_set and target_set have different sizes!")

Shuffled.
New shape: (50, 151)


In [21]:
dist_method = 0
r = 0.0
function_used = psdtw_prime_parallel_bsf

In [22]:
# Warmup for numba
start = time.time()
nearest_neighbor_search(query_set[0][:-1], target_set[:, :-1], r=r, l=l,  P=P, dist_method=dist_method, dist_func=function_used)
end = time.time()
elapsed_time = end - start
# print(elapsed_time)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [23]:
print("Starting nearest neighbor search over the entire query set...")
print(dataset_name)
all_count_dist_calls = []
start = time.time()
precision_at_1 = 0
for i in range(0, len(query_set)):
    query_vec = query_set[i][:-1]
    target_vecs = target_set[:, :-1]
    best_idx, bsf, total_dist_calls = nearest_neighbor_search(query_vec, target_vecs, r=r, l=l,  P=P, dist_method=dist_method, dist_func=function_used)
    all_count_dist_calls.append(total_dist_calls)
    
    # Check correctness using index
    if target_set[best_idx, -1] == query_set[i, -1]:
        precision_at_1 += 1
    # print(i, end=" ")
print()
print(
    f"{precision_at_1 / len(query_set):.2f}",
    end=" & ",
)
end = time.time()
elapsed_time = end - start
print()
print("Elapsed time: " + str(elapsed_time))
print("Average Elapsed time: " + str(elapsed_time / len(query_set)))

total_count_dist_calls = 0
for r in all_count_dist_calls:
    total_count_dist_calls += np.sum(r)
print("Total distance measure calls: " + str(total_count_dist_calls))
new_elapsed_time = elapsed_time
new_total_count_dist_calls = total_count_dist_calls

Starting nearest neighbor search over the entire query set...
GunPoint

0.96 & 
Elapsed time: 4.245957136154175
Average Elapsed time: 0.0849191427230835
Total distance measure calls: 51075577


# Calculate % distance calls pruned and Speedup

In [24]:
# pruned_dist_calls = original_total_count_dist_calls - new_total_count_dist_calls
# pruned_dist_calls_percentage = pruned_dist_calls / original_total_count_dist_calls * 100
# print(f"Pruned distance measure calls: {pruned_dist_calls} ({pruned_dist_calls_percentage:.2f}%)")
# # speed up
# speed_up = original_elapsed_time / new_elapsed_time
# print(f"Speed up: {speed_up:.2f}x")

# End

In [25]:
import datetime

print(f"This notebook was last run end-to-end on: {datetime.datetime.now()}\n")
###
###
###

This notebook was last run end-to-end on: 2026-03-19 14:24:11.554791



<!-- # Testing with PSD methods -->

In [26]:
from numba import njit, prange
import numpy as np
import math

@njit
def delta2(d, X_sorted):
    """Algorithm 1: Calculates the minimum squared distance between a point and a sorted array."""
    if len(X_sorted) == 0:
        return 0.0
    
    # Check boundaries first (necessity of binary search)
    if d <= X_sorted[0]:
        return (d - X_sorted[0]) ** 2
    elif d >= X_sorted[-1]:
        return (d - X_sorted[-1]) ** 2
    else:
        # Binary search for the closest point
        idx = np.searchsorted(X_sorted, d)
        dist1 = (d - X_sorted[idx - 1]) ** 2
        dist2 = (d - X_sorted[idx]) ** 2
        return min(dist1, dist2)

In [35]:
@njit
def delta(d, window_sorted):
    """
    Parameters
    ----------
    d : float
        Data point.
    window_sorted : array_like, shape (k,)
        Sorted data sequence.

    Returns
    -------
    float
        Squared distance between d and the closest point in window_sorted.
    """
    if window_sorted.size == 0:
        return 0.0  # should not happen if constructed correctly

    # Check the necessity of binary search
    # Fast boundary checks
    if d <= window_sorted[0]:
        diff = d - window_sorted[0]
        return diff * diff
    if d > window_sorted[-1]:
        diff = d - window_sorted[-1]
        return diff * diff

    # Do binary search if needed
    idx = np.searchsorted(window_sorted, d)
    d1 = d - window_sorted[idx]
    d2 = d - window_sorted[idx - 1]
    return min(d1 * d1, d2 * d2)
@njit
def insert_to_sorted(arr, val):
    idx = np.searchsorted(arr, val)
    res = np.empty(len(arr) + 1, dtype=arr.dtype) # res: result
    res[:idx] = arr[:idx]
    res[idx] = val
    res[idx+1:] = arr[idx:]
    return res

In [42]:
@njit(parallel=True)
def psdtw_prime_parallel_bsf_lb2_test(Q, C, r, l, P, dist_method, bsf=np.inf):
    # print("psdtw_prime_parallel_bsf_lb2_test")
    count_dist_calls = 0
    m = len(Q)
    n = len(C)
    assert m == n, "m should be equal to n"
    
    l_root = math.sqrt(l)
    L_Q_gavg = m / P
    L_Q_gmin = int(math.ceil(L_Q_gavg / l_root))
    L_Q_gmax = int(math.floor(L_Q_gavg * l_root))
    L_C_gavg = n / P
    L_C_gmin = int(math.ceil(L_C_gavg / l_root))
    L_C_gmax = int(math.floor(L_C_gavg * l_root))
    L_gmax = max(L_Q_gmax, L_C_gmax)

    D = np.full((m + 1, n + 1, P + 1), np.inf)
    D[0, 0, 0] = 0.0
    D_cut = np.full((m + 1, n + 1, P + 1, 2), -1, dtype=np.int64)

    # Outer loop 'p' must remain sequential because it depends on p-1
    for p in range(1, P + 1):
        
        # Parallelize the 'i' loop.
        for i in prange(L_Q_gmin * p, min(L_Q_gmax * p, m) + 1):
            # Initialize with correct type (list of sorted float arrays) for Numba inference
            cached_windows_sorted = [np.sort(Q[:0])]
            cached_r_int = -1
            for L_Q in range(L_Q_gmin, L_Q_gmax + 1):
                i_prime = i - L_Q
                if i_prime < 0:
                    continue
                Q_segment = Q[i_prime:i][::-1] # |Q_segement| = L_Q

                L_C_min = max(L_C_gmin, int(math.ceil(L_Q / l)))
                L_C_max = min(int(math.floor(L_Q * l)), L_C_gmax)

                r_int = int(r * max(len(Q_segment), L_C_max))

                windows_sorted = []
                # for k in range(1, L_C_max + 1):
                #     idx_start = int(max(1, math.ceil(k / l) - r_int)) - 1 # 1-based to 0-based index
                #     idx_end = int(min(math.floor(k * l) + r_int, len(Q_segment))) - 1 # 1-based to 0-based index
                    
                #     window = Q_segment[idx_start : idx_end + 1] # Inclusive of idx_end
                #     windows_sorted.append(np.sort(window))

                if cached_r_int == r_int:
                    for k in range(1, L_C_max + 1):
                        if k < len(cached_windows_sorted):
                            calc_end_1 = math.ceil(k * l) + r_int
                            prev_idx_end = int(min(calc_end_1, len(Q_segment) - 1)) - 1
                            new_idx_end = int(min(calc_end_1, len(Q_segment))) - 1
                            if new_idx_end > prev_idx_end:
                                new_window_sorted = insert_to_sorted(
                                    cached_windows_sorted[k],
                                    Q_segment[new_idx_end]
                                )
                                windows_sorted.append(new_window_sorted)
                            else:
                                windows_sorted.append(cached_windows_sorted[k])
                        else:
                            idx_start = int(max(1, math.ceil(k / l) - r_int)) - 1 # 1-based to 0-based index
                            idx_end = int(min(math.floor(k * l) + r_int, len(Q_segment))) - 1 # 1-based to 0-based index
                            window = Q_segment[idx_start : idx_end + 1] # Inclusive of idx_end
                            windows_sorted.append(np.sort(window))
                else:
                    for k in range(1, L_C_max + 1):
                        idx_start = int(max(1, math.ceil(k / l) - r_int)) - 1 # 1-based to 0-based index
                        idx_end = int(min(math.floor(k * l) + r_int, len(Q_segment))) - 1 # 1-based to 0-based index
                        window = Q_segment[idx_start : idx_end + 1] # Inclusive of idx_end
                        windows_sorted.append(np.sort(window))
                cached_windows_sorted = windows_sorted
                cached_r_int = r_int

                for j in range(L_C_gmin * p, min(L_C_gmax * p, n) + 1):
                    for L_C in range(L_C_min, L_C_max + 1):
                        j_prime = j - L_C
                        if j_prime < 0:
                            continue
                        D_cost = D[i_prime, j_prime, p - 1]

                        C_segment = C[j_prime:j][::-1] # |C_segment| = L_C
                        
                        if L_C == L_C_min:
                            lb = (Q_segment[0] - C_segment[0]) ** 2
                            for k in range(1, L_C):
                                lb += delta(C_segment[k], windows_sorted[k])
                        else:
                            lb += delta(C_segment[L_C - 1], windows_sorted[L_C - 1])
                        # Use lb with the last point to further tighten the bound
                        lb_check = lb - delta(C_segment[L_C - 1], windows_sorted[L_C - 1]) + (Q_segment[-1] - C_segment[-1]) ** 2

                        if D_cost + lb_check > D[i, j, p]:
                            continue
                        
                        if np.isinf(D_cost):
                            continue
                        if D_cost > bsf:
                            continue
                        if D_cost > D[i, j, p]: # D[i][j][p] stores the best_so_far
                            continue
                        
                        dist_cost = usdtw_prime(
                            Q_segment,
                            C_segment,
                            r=r,
                            L=L_gmax, # For upsampling
                            dist_method=dist_method,
                        )
                        
                        count_dist_calls += 1
                        
                        cur_cost = D_cost + dist_cost
                        if cur_cost < D[i, j, p]:
                            D[i, j, p] = cur_cost
                            D_cut[i, j, p, 0] = i_prime
                            D_cut[i, j, p, 1] = j_prime

    cuts = np.zeros((P, 4), dtype=np.int64)
    i, j, p = m, n, P
    while p > 0:
        i_prime = D_cut[i, j, p, 0]
        j_prime = D_cut[i, j, p, 1]
        cuts[p - 1, 0] = i_prime
        cuts[p - 1, 1] = i
        cuts[p - 1, 2] = j_prime
        cuts[p - 1, 3] = j
        i, j, p = i_prime, j_prime, p - 1
        
    return D[m, n, P], count_dist_calls, cuts

In [29]:
Q = query_set[0][:-1]
C = target_set[1][:-1]

r =0.1
P = 2
l = 1.50
dist_method = 1

In [30]:
# Warmup for numba
psdtw_prime_parallel(Q, C, r=r, l=l, P=P, dist_method=dist_method)

start = time.time()
res= psdtw_prime_parallel(Q, C, r=r, l=l, P=P, dist_method=dist_method)
end = time.time()
print(res)
print(f"Elapsed time: {end - start} seconds")

(0.8950274893486205, 94169, array([[  0,  72,   0,  65],
       [ 72, 150,  65, 150]]))
Elapsed time: 0.26511693000793457 seconds


In [31]:
# Warmup for numba
psdtw_prime_parallel_bsf_lb(Q, C, r=r, l=l, P=P, dist_method=dist_method)

start = time.time()
res = psdtw_prime_parallel_bsf_lb(Q, C, r=r, l=l, P=P, dist_method=dist_method)
end = time.time()
print(res)
print(f"Elapsed time: {end - start} seconds")

(0.8950274893486205, 88670, array([[  0,  72,   0,  65],
       [ 72, 150,  65, 150]]))
Elapsed time: 0.2967040538787842 seconds


In [41]:
# Warmup for numba
psdtw_prime_parallel_bsf_lb2_test(Q, C, r=r, l=l, P=P, dist_method=dist_method)

start = time.time()
res = psdtw_prime_parallel_bsf_lb2_test(Q, C, r=r, l=l, P=P, dist_method=dist_method)
end = time.time()
print(res)
print(f"Elapsed time: {end - start} seconds")

(0.8950274893486205, 89412, array([[  0,  72,   0,  65],
       [ 72, 150,  65, 150]]))
Elapsed time: 0.2721562385559082 seconds


In [ ]:
P = 2
l = 1.50
dist_method = 1
r = 0.1

num_mismatches = 0
max_diff = 0.0
tolerance = 1e-6

print(f"Comparing psdtw_prime_parallel and psdtw_prime_parallel_bsf_lb2 for {len(query_set)} queries against {len(target_set)} targets...")
print(f"Parameters: r={r}, l={l}, P={P}, dist_method={dist_method}")

# Iterate through all query and target pairs
# To avoid taking too long if datasets are huge, we can limit or print progress
for i in range(len(query_set)):
    Q = query_set[i][:-1]
    
    # Optional: check fewer targets if it's too slow, but user asked for "all combination"
    # For now, let's try all.
    for j in range(len(target_set)):
        C = target_set[j][:-1]
        
        # Calculate distance using method 1
        dist1, _, _ = psdtw_prime_parallel(Q, C, r=r, l=l, P=P, dist_method=dist_method)

        # Calculate distance using method 2
        dist2, _, _ = psdtw_prime_parallel_bsf_lb2_test(Q, C, r=r, l=l, P=P, dist_method=dist_method, bsf=np.inf)
        
        diff = abs(dist1 - dist2)
        if diff > max_diff:
            max_diff = diff
            
        if diff > tolerance:
            num_mismatches += 1
            if num_mismatches <= 5: # Print first few mismatches
                 print(f"Mismatch at Query {i}, Target {j}: {dist1} vs {dist2} (diff: {diff})")

print(f"Total mismatches: {num_mismatches}")
print(f"Max difference: {max_diff}")
if num_mismatches == 0:
    print("Test Passed: Results are similar for all combinations.")
else:
    print("Test Failed: Mismatches found.")

Comparing psdtw_prime_parallel and psdtw_prime_parallel_bsf_lb2 for 50 queries against 50 targets...
Parameters: r=0.1, l=1.5, P=2, dist_method=1


SystemError: CPUDispatcher(<function psdtw_prime_parallel at 0x162d7e5c0>) returned a result with an exception set